In [2]:
import pandas as pd

In [3]:
particle_df = pd.read_parquet("../../../data/processed/particle_labeled.parquet")
print(f"Shape: {particle_df.shape}")
print(f"Columns: {particle_df.columns}")

Shape: (2801667, 95)
Columns: Index(['stub_id', 'particle_id', 'relevance_class', 'ac', 'ag', 'al', 'ar',
       'as', 'at', 'au', 'b', 'ba', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co',
       'cr', 'cs', 'cu', 'dy', 'er', 'eu', 'f', 'fe', 'fr', 'ga', 'gd', 'ge',
       'hf', 'hg', 'ho', 'i', 'in', 'ir', 'k', 'kr', 'la', 'lu', 'mg', 'mn',
       'mo', 'n', 'na', 'nb', 'nd', 'ne', 'ni', 'np', 'o', 'os', 'p', 'pa',
       'pb', 'pd', 'pm', 'po', 'pr', 'pt', 'pu', 'ra', 'rb', 're', 'rh', 'rn',
       'ru', 's', 'sb', 'sc', 'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb', 'tc',
       'te', 'th', 'ti', 'tl', 'tm', 'u', 'v', 'w', 'xe', 'y', 'yb', 'zn',
       'zr', 'merged_relevance_class', 'final_class', 'label'],
      dtype='str')


In [10]:
# Step 1: Drop non-informative elements
meta_cols = [
    "stub_id", "particle_id", "relevance_class", "merged_relevance_class", "final_class", "label"
    ]
print(f"# of non-element columns: {len(meta_cols)}")

element_cols = [c for c in particle_df.columns if c not in meta_cols]
print(f"\n# of original element columns: {len(element_cols)}")

# non-zero rates
nonzero_rates = (particle_df[element_cols] > 0).mean().sort_values(ascending=False)

# informative elements
informative_elements = nonzero_rates[nonzero_rates > 0.01].index.tolist()
print(f"\n# of informative elements: {len(informative_elements)}")

informative_df = particle_df[meta_cols + informative_elements]
print(f"\nDF shape after dropping non-informative elements: {informative_df.shape}")

# of non-element columns: 6

# of original element columns: 89

# of informative elements: 27

DF shape after dropping non-informative elements: (2801667, 33)


In [33]:
# Step 2: Drop ambiguous particles 
binary_informative_df = informative_df[informative_df["label"] != "Ambiguous"]

In [34]:
# Step 3: Add target column to encode 'label' (GSR=1 / Non-GSR=0)
binary_informative_df["target"] = (binary_informative_df["label"] == "GSR").astype(int)
binary_informative_df.shape

(2294985, 34)

In [35]:
# Write pre-processed dataset
binary_informative_df.to_parquet("../../../data/processed/preprocessed.parquet")

# Write ambiguous dataset for future use
ambiguous_df = informative_df[informative_df["label"] == "Ambiguous"]
ambiguous_df.to_parquet("../../../data/processed/particle_ambiguous.parquet")

In [36]:
binary_informative_df.columns

Index(['stub_id', 'particle_id', 'relevance_class', 'merged_relevance_class',
       'final_class', 'label', 'o', 's', 'cu', 'ba', 'al', 'si', 'ca', 'pb',
       'sb', 'fe', 'zn', 'cl', 'k', 'na', 'mg', 'ti', 'sn', 'p', 'mn', 'as',
       'cr', 'br', 'mo', 'sr', 'ni', 'w', 'hg', 'target'],
      dtype='str')